In [1]:
import os
import glob
import numpy as np
import pandas as pd

# Caminhos do repositório
PASTA_BRUTOS = os.path.join("..", "dados", "sujo")
PASTA_LIMPOS = os.path.join("..", "dados", "limpo")
os.makedirs(PASTA_LIMPOS, exist_ok=True)

arquivos_csv = sorted(glob.glob(os.path.join(PASTA_BRUTOS, "ca-*.csv")))
print(f"Arquivos encontrados ({len(arquivos_csv)}):", [os.path.basename(a) for a in arquivos_csv])

Arquivos encontrados (6): ['ca-2018-01.csv', 'ca-2018-02.csv', 'ca-2019-01.csv', 'ca-2019-02.csv', 'ca-2020-01.csv', 'ca-2020-02.csv']


In [2]:
dfs = []

for caminho in arquivos_csv:
    nome_arquivo = os.path.basename(caminho)
    df_part = pd.read_csv(
        caminho,
        sep=";",
        dtype=str,
        encoding="utf-8-sig",
        quotechar='"'
    )
    assert df_part.shape[1] == 16, f"Colunas inválidas em {nome_arquivo}: {df_part.shape[1]}"
    df_part["arquivo_origem"] = nome_arquivo
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
print(f"Total de registros extraídos: {len(df):,}")
df.head(3)

Total de registros extraídos: 2,686,331


,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira,arquivo_origem
0,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,GNV,02/01/2018,"2,37","1,7383",R$ / m³,PETROBRAS DISTRIBUIDORA S.A.,ca-2018-01.csv
1,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,DIESEL S10,02/01/2018,"3,24","3,1366",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.,ca-2018-01.csv
2,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,ETANOL,02/01/2018,"2,93","2,6194",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.,ca-2018-01.csv


In [3]:
colunas_texto = [c for c in df.columns if c != "arquivo_origem"]

for col in colunas_texto:
    df[col] = df[col].str.strip().replace("", pd.NA)

print("Contagem de nulos após trim:")
print(df[colunas_texto].isna().sum())

Contagem de nulos após trim:
Regiao - Sigla             0
Estado - Sigla             0
Municipio                  0
Revenda                    0
CNPJ da Revenda            0
Nome da Rua                0
Numero Rua              2067
Complemento          1996986
Bairro                  9353
Cep                        0
Produto                    0
Data da Coleta             0
Valor de Venda             0
Valor de Compra      1761960
Unidade de Medida          0
Bandeira                   0
dtype: int64


In [4]:
df["cnpj"] = df["CNPJ da Revenda"].str.replace(r"\D", "", regex=True)
df["cep"] = df["Cep"].str.replace(r"\D", "", regex=True)

regex_sn = r"^(?:S\s*/?\s*N\.?\s*[º°O]?|SNR|SN)$"
df["numero_rua"] = (
    df["Numero Rua"]
    .fillna("S/N")
    .str.upper()
    .str.replace(regex_sn, "S/N", regex=True)
)

assert (df["cnpj"].str.len() == 14).all(), "CNPJs fora do padrão de 14 dígitos!"
assert (df["cep"].str.len() == 8).all(), "CEPs fora do padrão de 8 dígitos!"
print("CNPJ, CEP e numero_rua validados com sucesso.")

CNPJ, CEP e numero_rua validados com sucesso.


In [5]:
df["data_coleta"] = pd.to_datetime(df["Data da Coleta"], format="%d/%m/%Y", errors="coerce")

df["valor_venda"] = pd.to_numeric(
    df["Valor de Venda"].str.replace(",", ".", regex=False),
    errors="coerce"
)
df["valor_compra"] = pd.to_numeric(
    df["Valor de Compra"].str.replace(",", ".", regex=False),
    errors="coerce"
)

assert df["data_coleta"].isna().sum() == 0, "Existem datas inválidas!"
assert (df["valor_venda"] > 0).all(), "Existem valores de venda <= 0!"


In [6]:
caminho_de_para = os.path.join("..", "dados", "de_para_bandeira.csv")

if os.path.exists(caminho_de_para):
    df_de_para = pd.read_csv(caminho_de_para, sep=";", dtype=str, encoding="utf-8-sig")
    mapa_bandeira = dict(zip(df_de_para["bandeira_original"], df_de_para["bandeira_padronizada"]))
else:
    mapa_bandeira = {
        "PETROX DISTRIBUIDORA": "PETROX",
        "ALE COMBUSTÍVEIS": "ALESAT",
    }

df["bandeira_original"] = df["Bandeira"]
df["bandeira_padronizada"] = df["Bandeira"].replace(mapa_bandeira)

print(
    f"Bandeiras distintas: {df['bandeira_original'].nunique()} originais -> "
    f"{df['bandeira_padronizada'].nunique()} padronizadas"
)

Bandeiras distintas: 73 originais -> 71 padronizadas


In [7]:
df["margem"] = np.where(
    df["valor_compra"].notna(),
    (df["valor_venda"] - df["valor_compra"]).round(4),
    np.nan
)

df["fl_compra_maior_venda"] = (
    df["valor_compra"].notna() & (df["valor_compra"] > df["valor_venda"])
)

df["ano_mes"] = df["data_coleta"].dt.to_period("M")
grupo_prod_mes = df.groupby(["Produto", "ano_mes"])["valor_venda"]
q1 = grupo_prod_mes.transform("quantile", 0.25)
q3 = grupo_prod_mes.transform("quantile", 0.75)
iqr = q3 - q1

df["fl_outlier"] = (
    (df["valor_venda"] < (q1 - 3 * iqr)) |
    (df["valor_venda"] > (q3 + 3 * iqr))
)

qtd_duplicatas = df.duplicated(subset=["cnpj", "Produto", "data_coleta"]).sum()
print(f"Duplicatas na chave (cnpj, Produto, data_coleta): {qtd_duplicatas}")
print(f"fl_compra_maior_venda = True: {df['fl_compra_maior_venda'].sum():,}")
print(f"fl_outlier = True: {df['fl_outlier'].sum():,}")
print("\nUnidades distintas por Produto:\n", df.groupby("Produto")["Unidade de Medida"].nunique())

Duplicatas na chave (cnpj, Produto, data_coleta): 0
fl_compra_maior_venda = True: 1,549
fl_outlier = True: 4,075

Unidades distintas por Produto:
 Produto
DIESEL                1
DIESEL S10            1
ETANOL                1
GASOLINA              1
GASOLINA ADITIVADA    1
GNV                   1
Name: Unidade de Medida, dtype: int64


In [12]:
# dim_tempo
datas = pd.date_range("2018-01-01", "2020-12-31", freq="D")
meses_pt = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
dias_pt = {
    0: "Segunda-feira", 1: "Terça-feira", 2: "Quarta-feira",
    3: "Quinta-feira", 4: "Sexta-feira", 5: "Sábado", 6: "Domingo"
}

dim_tempo = pd.DataFrame({"data": datas})
dim_tempo["sk_tempo"] = dim_tempo["data"].dt.strftime("%Y%m%d").astype(int)
dim_tempo["dia"] = dim_tempo["data"].dt.day
dim_tempo["mes"] = dim_tempo["data"].dt.month
dim_tempo["nome_mes"] = dim_tempo["mes"].map(meses_pt)
dim_tempo["trimestre"] = dim_tempo["data"].dt.quarter
dim_tempo["semestre"] = np.where(dim_tempo["mes"] <= 6, 1, 2)
dim_tempo["ano"] = dim_tempo["data"].dt.year
dim_tempo["dia_semana"] = dim_tempo["data"].dt.dayofweek.map(dias_pt)

condicoes = [
    dim_tempo["data"] < "2018-05-21",
    (dim_tempo["data"] >= "2018-05-21") & (dim_tempo["data"] <= "2018-06-05"),
    (dim_tempo["data"] > "2018-06-05") & (dim_tempo["data"] < "2020-03-11"),
    dim_tempo["data"] >= "2020-03-11"
]
periodos = ["Pré-Greve 2018", "Greve Caminhoneiros 2018", "Normalidade 2018-2019", "Pandemia 2020"]
dim_tempo["periodo_historico"] = np.select(condicoes, periodos, default="Outro")

# dim_posto (cadastro mais recente por CNPJ)
dim_posto = (
    df.sort_values("data_coleta")
    .drop_duplicates(subset=["cnpj"], keep="last")
    [["cnpj", "Revenda", "Nome da Rua", "numero_rua", "Complemento", "Bairro", "cep"]]
    .rename(columns={
        "Revenda": "razao_social",
        "Nome da Rua": "rua",
        "Complemento": "complemento",
        "Bairro": "bairro"
    })
    .reset_index(drop=True)
)
dim_posto.insert(0, "sk_posto", dim_posto.index + 1)

# dim_localizacao
dim_localizacao = (
    df[["Regiao - Sigla", "Estado - Sigla", "Municipio"]]
    .drop_duplicates()
    .rename(columns={"Regiao - Sigla": "regiao", "Estado - Sigla": "uf", "Municipio": "municipio"})
    .sort_values(["regiao", "uf", "municipio"])
    .reset_index(drop=True)
)
dim_localizacao.insert(0, "sk_localizacao", dim_localizacao.index + 1)

# dim_produto
dim_produto = (
    df[["Produto", "Unidade de Medida"]]
    .drop_duplicates()
    .rename(columns={"Produto": "produto", "Unidade de Medida": "unidade_medida"})
    .sort_values("produto")
    .reset_index(drop=True)
)
dim_produto.insert(0, "sk_produto", dim_produto.index + 1)

# dim_bandeira
dim_bandeira = (
    df[["bandeira_original", "bandeira_padronizada"]]
    .drop_duplicates()
    .sort_values(["bandeira_padronizada", "bandeira_original"])
    .reset_index(drop=True)
)
dim_bandeira["fl_branca"] = dim_bandeira["bandeira_padronizada"] == "BRANCA"
dim_bandeira.insert(0, "sk_bandeira", dim_bandeira.index + 1)

# fato_preco
fato_preco = (
    df.merge(dim_tempo[["data", "sk_tempo"]], left_on="data_coleta", right_on="data", how="left")
      .merge(dim_posto[["cnpj", "sk_posto"]], on="cnpj", how="left")
      .merge(
          dim_localizacao,
          left_on=["Regiao - Sigla", "Estado - Sigla", "Municipio"],
          right_on=["regiao", "uf", "municipio"],
          how="left"
      )
      .merge(
          dim_produto,
          left_on=["Produto", "Unidade de Medida"],
          right_on=["produto", "unidade_medida"],
          how="left"
      )
      .merge(
          dim_bandeira[["bandeira_original", "bandeira_padronizada", "sk_bandeira"]],
          on=["bandeira_original", "bandeira_padronizada"],
          how="left"
      )
      [[
          "sk_tempo", "sk_posto", "sk_produto", "sk_bandeira", "sk_localizacao",
          "valor_venda", "valor_compra", "margem",
          "fl_compra_maior_venda", "fl_outlier", "arquivo_origem"
      ]]
)

tabelas_dw = {
    "dim_tempo": dim_tempo,
    "dim_posto": dim_posto,
    "dim_localizacao": dim_localizacao,
    "dim_produto": dim_produto,
    "dim_bandeira": dim_bandeira,
    "fato_preco": fato_preco,
}

for nome, tabela in tabelas_dw.items():
    print(f"{nome}: {tabela.shape[0]:,} linhas x {tabela.shape[1]} colunas")

C:\Users\vinic\AppData\Local\Temp\ipykernel_6676\682758653.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dim_tempo["dia_semana"] = dim_tempo["data"].dt.dayofweek.map(dias_pt)


dim_tempo: 1,096 linhas x 10 colunas
dim_posto: 19,275 linhas x 8 colunas
dim_localizacao: 474 linhas x 4 colunas
dim_produto: 6 linhas x 3 colunas
dim_bandeira: 73 linhas x 4 colunas
fato_preco: 2,686,331 linhas x 11 colunas
